# cctv_dfd — train + evaluate on FaceForensics++ (c23, 200+200)

End-to-end Colab notebook. Outputs you can pull off Colab:
- `ffpp_dinov2.pt` — the trained head checkpoint (drop into your laptop's `cctv_dfd/results/checkpoints/`)
- `ffpp_results.zip` — metrics JSON + charts + cross-dataset eval table
- `ffpp_test_samples.zip` — 25 real + 25 fake face crops from the FF++ test split, for drag-and-drop demos in the local UI

**Total runtime on T4:** ~70-90 min wall clock (FF++ download dominates).

**Each step is idempotent** — if Colab disconnects mid-run, re-running the notebook resumes (files already on disk are skipped).

## 1. Setup — GPU, clone, install

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
import os
if not os.path.isdir('/content/repo'):
    !git clone https://github.com/Aditya1404Sal/Deepfake-cctv.git /content/repo
else:
    !cd /content/repo && git pull
%cd /content/repo/cctv_dfd

In [ ]:
!pip install -q -e .
!pip install -q facenet-pytorch

## 2. Download FaceForensics++ (c23, 200 videos per class)

Pulls 200 real + 200 each of Deepfakes / Face2Face / FaceSwap / NeuralTextures, all at c23 compression. ~13 GB total, ~25-40 min depending on TUM server load. Uses `--server EU2` per the access email.

Pre-accepts the FF++ EULA via `yes |` because the script prompts on stdin.

In [ ]:
%cd /content
if not os.path.exists('ffpp_download.py'):
    !wget -q https://kaldir.vc.in.tum.de/faceforensics_download_v4.py -O ffpp_download.py
!ls -la ffpp_download.py

In [ ]:
# Real (original YouTube source videos)
!yes | python ffpp_download.py /content/ffpp \
    --dataset original \
    --compression c23 \
    --num_videos 200 \
    --server EU2

In [ ]:
# Deepfakes
!yes | python ffpp_download.py /content/ffpp \
    --dataset Deepfakes \
    --compression c23 \
    --num_videos 200 \
    --server EU2

In [ ]:
# Face2Face
!yes | python ffpp_download.py /content/ffpp \
    --dataset Face2Face \
    --compression c23 \
    --num_videos 200 \
    --server EU2

In [ ]:
# FaceSwap
!yes | python ffpp_download.py /content/ffpp \
    --dataset FaceSwap \
    --compression c23 \
    --num_videos 200 \
    --server EU2

In [ ]:
# NeuralTextures
!yes | python ffpp_download.py /content/ffpp \
    --dataset NeuralTextures \
    --compression c23 \
    --num_videos 200 \
    --server EU2

In [ ]:
# Sanity check: should be ~1000 mp4 files total
import subprocess
n = subprocess.check_output(['bash', '-lc', 'find /content/ffpp -name "*.mp4" | wc -l']).decode().strip()
print(f'Total mp4s: {n}')
!du -sh /content/ffpp

## 3. Extract face crops from videos

Per video: sample evenly-spaced frames → detect strongest face → 1.3× margin crop → resize to 224×224 JPEG. **Real**: 20 frames × 200 videos = ~4K crops. **Fake**: 5 frames × 200 videos × 4 methods = ~4K crops. Class-balanced.

Skips faces it can't detect, so output counts are slightly under the maximum.

In [ ]:
%cd /content/repo/cctv_dfd
from pathlib import Path
import cv2
from tqdm.auto import tqdm
from cctv_dfd.face_extract import _FaceDetector, _expand_bbox, _resize_square, _save_jpeg, TARGET_SIZE, MARGIN

detector = _FaceDetector()

def sample_frame_indices(total: int, n: int):
    if total < 1 or n < 1:
        return []
    return [int(total * i / n) for i in range(n)]

def extract_video(video_path: Path, out_dir: Path, label: str, frames: int = 20) -> int:
    cap = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    written = 0
    for idx in sample_frame_indices(total, frames):
        out_path = out_dir / label / f'{video_path.stem}_f{idx:05d}.jpg'
        if out_path.exists():  # idempotent resume
            written += 1
            continue
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, frame = cap.read()
        if not ok:
            continue
        bbox = detector.detect_best(frame)
        if bbox is None:
            continue
        nb = _expand_bbox(bbox, frame.shape[:2], MARGIN)
        crop = frame[nb[1]:nb[3], nb[0]:nb[2]]
        if crop.size == 0:
            continue
        crop = _resize_square(crop, TARGET_SIZE)
        _save_jpeg(crop, out_path)
        written += 1
    cap.release()
    return written

FFPP_ROOT = Path('/content/ffpp')
OUT = Path('/content/repo/cctv_dfd/data/processed/ffpp')

real_videos = sorted((FFPP_ROOT / 'original_sequences/youtube/c23/videos').glob('*.mp4'))
fake_dirs = [
    FFPP_ROOT / 'manipulated_sequences/Deepfakes/c23/videos',
    FFPP_ROOT / 'manipulated_sequences/Face2Face/c23/videos',
    FFPP_ROOT / 'manipulated_sequences/FaceSwap/c23/videos',
    FFPP_ROOT / 'manipulated_sequences/NeuralTextures/c23/videos',
]
fake_videos = []
for d in fake_dirs:
    fake_videos.extend(sorted(d.glob('*.mp4')))

print(f'Real videos: {len(real_videos)}')
print(f'Fake videos: {len(fake_videos)}  (across 4 methods)')

In [ ]:
# Extract — runs ~30 min on T4 the first time. Resumable.
real_count = 0
for v in tqdm(real_videos, desc='Real videos'):
    real_count += extract_video(v, OUT, 'Real', frames=20)
fake_count = 0
for v in tqdm(fake_videos, desc='Fake videos'):
    fake_count += extract_video(v, OUT, 'Fake', frames=5)

print(f'Real crops:  {real_count}')
print(f'Fake crops:  {fake_count}')
!ls data/processed/ffpp/Real | wc -l
!ls data/processed/ffpp/Fake | wc -l

## 4. Train DINOv2 + MLP head on FF++

In [ ]:
# Clean any prior FF++ artifacts so the run is reproducible
!rm -f results/checkpoints/ffpp_dinov2.pt results/metrics/ffpp_dinov2*.json

In [ ]:
# 20 epochs, early stop on val AUC. ~15-20 min on T4.
!python -m cctv_dfd.cli train \
    --config configs/dinov2s_mlp.yaml \
    --dataset ffpp --epochs 20 --batch-size 64 --tag ffpp_dinov2

In [ ]:
# Show the best-epoch summary
import json, pathlib
tr = json.loads(pathlib.Path('results/metrics/ffpp_dinov2.json').read_text())
print(f"best epoch:    {tr['best_epoch']}")
print(f"best val_auc:  {tr['best_val_auc']:.4f}")
print(f"epochs ran:    {len(tr['history'])}")
print(f"elapsed:       {tr['elapsed_sec']/60:.1f} min")

## 5. Cross-dataset evaluation

Run the FF++-trained checkpoint over both:
- The FF++ test split (in-domain)
- The existing local 10K dataset (out-of-domain) — only if the local data is present from the earlier 01_train_local notebook

In [ ]:
# Always run FF++ in-domain eval
datasets_to_eval = ['ffpp']

# If local data is around, include it as cross-dataset OOD eval
if pathlib.Path('data/processed/local/Real').exists():
    datasets_to_eval.append('local')
    print('local dataset present — will run cross-dataset eval too')
else:
    print('local dataset not present — skipping cross-dataset eval row')
print('datasets to eval:', datasets_to_eval)

In [ ]:
args = ' '.join(datasets_to_eval)
!python -m cctv_dfd.cli eval \
    --run-tag ffpp_dinov2 --datasets {args} \
    --profiles all --enhance-modes none forensic

In [ ]:
import json, pathlib
data = json.loads(pathlib.Path('results/metrics/ffpp_dinov2_eval.json').read_text())
rows = data['rows']
print(f"{'dataset':8s} {'profile':16s} {'enhance':9s} {'acc':>6s} {'prec':>6s} {'rec':>6s} {'f1':>6s} {'auc':>6s} {'eer':>6s} {'n':>5s}")
print('-' * 80)
for r in rows:
    print(f"{r['dataset']:8s} {r['profile']:16s} {r['enhance']:9s} "
          f"{r['acc']:6.3f} {r['precision']:6.3f} {r['recall']:6.3f} "
          f"{r['f1']:6.3f} {r['auc']:6.3f} {r['eer']:6.3f} {r['n']:5d}")

## 6. Charts for the report

In [ ]:
import matplotlib.pyplot as plt
from collections import defaultdict
PROFILES = ['clean', 'light_cctv', 'heavy_cctv', 'low_light_gray', 'ir_pseudo']

# AUC by profile, one line per (dataset, enhance) pair
fig, ax = plt.subplots(figsize=(9, 5))
by_key = defaultdict(list)
for r in rows:
    by_key[(r['dataset'], r['enhance'])].append((r['profile'], r['auc']))
for (ds, enh), pairs in sorted(by_key.items()):
    d = dict(pairs)
    ax.plot(PROFILES, [d.get(p, float('nan')) for p in PROFILES],
            marker='o', label=f'{ds} / {enh}')
ax.set_xlabel('degradation profile'); ax.set_ylabel('AUC-ROC')
ax.set_ylim(0.5, 1.0); ax.grid(alpha=0.3); ax.legend()
ax.set_title('AUC across CCTV degradation profiles — FF++ trained, evaluated cross-dataset')
plt.tight_layout()
plt.savefig('results/metrics/ffpp_auc_by_profile.png', dpi=120)
plt.show()

In [ ]:
# Training curves
tr = json.loads(pathlib.Path('results/metrics/ffpp_dinov2.json').read_text())
hist = tr['history']
epochs = [h['epoch'] for h in hist]
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(epochs, [h['train_loss'] for h in hist], label='train loss', marker='o')
ax.plot(epochs, [h['val_loss'] for h in hist], label='val loss', marker='s')
ax2 = ax.twinx()
ax2.plot(epochs, [h['val_auc'] for h in hist], color='tab:green', label='val AUC', marker='^')
ax.set_xlabel('epoch'); ax.set_ylabel('loss'); ax2.set_ylabel('AUC')
ax.legend(loc='upper left'); ax2.legend(loc='lower right')
ax.set_title(f"FF++ training curves (best epoch={tr['best_epoch']}, best AUC={tr['best_val_auc']:.3f})")
plt.tight_layout()
plt.savefig('results/metrics/ffpp_training_curves.png', dpi=120)
plt.show()

## 7. Bundle test samples (real + fake) for local UI demos

Pulls 25 real + 25 fake face crops from the FF++ **test split** (deterministic, never seen during training). These go in a zip you can drop into the local web UI for live demos.

In [ ]:
from cctv_dfd.data import list_items, resolve_dataset_dir, train_val_test_split, LABEL_REAL, LABEL_FAKE
import shutil

items = list_items(resolve_dataset_dir('ffpp'))
_, _, test_items = train_val_test_split(items, seed=1337)
test_real = [it for it in test_items if it.label == LABEL_REAL][:25]
test_fake = [it for it in test_items if it.label == LABEL_FAKE][:25]
print(f'real samples: {len(test_real)} | fake samples: {len(test_fake)}')

sample_dir = pathlib.Path('/content/ffpp_test_samples')
if sample_dir.exists():
    shutil.rmtree(sample_dir)
(sample_dir / 'Real').mkdir(parents=True)
(sample_dir / 'Fake').mkdir(parents=True)
for it in test_real:
    shutil.copy(it.path, sample_dir / 'Real' / it.path.name)
for it in test_fake:
    shutil.copy(it.path, sample_dir / 'Fake' / it.path.name)

!ls /content/ffpp_test_samples/Real | head
!ls /content/ffpp_test_samples/Fake | head

In [ ]:
# Zip the test samples
!cd /content && zip -qr /content/ffpp_test_samples.zip ffpp_test_samples/
!ls -lh /content/ffpp_test_samples.zip

## 8. Bundle the full results zip

In [ ]:
!cd /content/repo/cctv_dfd && zip -qr /content/ffpp_results.zip \
    results/checkpoints/ffpp_dinov2.pt \
    results/metrics/ffpp_dinov2.json \
    results/metrics/ffpp_dinov2_eval.json \
    results/metrics/ffpp_dinov2_eval.md \
    results/metrics/ffpp_auc_by_profile.png \
    results/metrics/ffpp_training_curves.png
!ls -lh /content/ffpp_results.zip
!unzip -l /content/ffpp_results.zip

## 9. Download to your laptop

Pulls the three things you need:
- **`ffpp_dinov2.pt`** (~400 KB) — the trained head, drop into your laptop's `cctv_dfd/results/checkpoints/`
- **`ffpp_results.zip`** (~2 MB) — all metrics + charts for the report
- **`ffpp_test_samples.zip`** (~3 MB) — 25 real + 25 fake test crops for local UI demos

**Run this cell IMMEDIATELY after training finishes — don't wait. Colab disconnects lose everything in /content/.**

In [ ]:
from google.colab import files
files.download('/content/repo/cctv_dfd/results/checkpoints/ffpp_dinov2.pt')
files.download('/content/ffpp_results.zip')
files.download('/content/ffpp_test_samples.zip')

## 10. On your laptop — swap the model and test

```bash
cd ~/Desktop/Deepfake-detection

# 1. move the new checkpoint into place
mv ~/Downloads/ffpp_dinov2.pt cctv_dfd/results/checkpoints/ffpp_dinov2.pt

# 2. unzip the test samples somewhere convenient (NOT inside the repo data folder)
mkdir -p ~/demo_samples
unzip ~/Downloads/ffpp_test_samples.zip -d ~/demo_samples/

# 3. unzip the results next to the existing local ones
unzip -o ~/Downloads/ffpp_results.zip -d cctv_dfd/

# 4. restart the FastAPI server pointing at the new checkpoint
cd cctv_dfd
source .venv/bin/activate
CCTV_DFD_CHECKPOINT=results/checkpoints/ffpp_dinov2.pt uvicorn api.server:app --host 0.0.0.0 --port 8000
```

Then open `http://localhost:3000/` (Vite frontend) and drag-drop any image from `~/demo_samples/ffpp_test_samples/Real/` and `~/demo_samples/ffpp_test_samples/Fake/`. Verdict should match the folder name with high confidence.